In [19]:
import sys
import os
import json
import pandas as pd
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import classification_report

import joblib

sys.path.append(r"C:\Users\germa\Desktop\Ejercicios\Agente-ML-IA-FugaClientes-")
from data.repositorio_cliente import generate_random_customer

db = [generate_random_customer() for i in range (500)]
dataf = pd.DataFrame(db)
# print(json.dumps(db, indent=4, default=str))
print(dataf)

     customer_id signup_date last_login_date        plan  monthly_fee  \
0           1052  2024-12-18      2025-10-27       basic        41.85   
1           4399  2024-12-29      2025-09-12       basic        48.85   
2           2358  2024-06-29      2025-09-30         pro        25.85   
3           6701  2025-12-07      2025-12-24         pro        12.32   
4           1847  2021-08-27      2025-07-31  enterprise        19.31   
..           ...         ...             ...         ...          ...   
495         1551  2023-01-29      2026-04-23         pro        12.78   
496         8034  2026-01-24      2026-05-07  enterprise        38.15   
497         3066  2025-08-25      2025-08-01         pro        10.41   
498         4293  2022-05-13      2025-06-08  enterprise        44.70   
499         9029  2024-12-21      2025-09-06       basic        31.13   

     support_tickets  usage_minutes country  is_active  
0                  0            120      AR          0  
1        

In [20]:
def preprocesado_datos(dataf):
    today = datetime.today()
    # dataf[["signup_year","signup_M","signup_Day"]] = dataf["signup_date"].str.split('-', expand=True)
    # dataf[["last_login_year","last_login_M","last_login_Day"]] = dataf["last_login_date"].str.split('-', expand=True)

    dataf["signup_date"] = pd.to_datetime(dataf["signup_date"])
    dataf["days_since_signup"] = (today - dataf["signup_date"]).dt.days

    # ---- last_login_date ----
    dataf["last_login_date"] = pd.to_datetime(dataf["last_login_date"])
    dataf["days_since_last_login"] = (today - dataf["last_login_date"]).dt.days


    # col_obj = dataf.select_dtypes(include=object).columns
    # dataf[col_obj] = dataf[col_obj].astype('category')

    dataf = dataf.drop(columns=["signup_date","last_login_date","customer_id"])
    return dataf
    
data_pre = preprocesado_datos(dataf)
data_pre.head(5)


,plan,monthly_fee,support_tickets,usage_minutes,country,is_active,days_since_signup,days_since_last_login
0,basic,41.85,0,120,AR,0,534,221
1,basic,48.85,15,402,AR,0,523,266
2,pro,25.85,3,698,US,1,706,248
3,pro,12.32,4,790,US,1,180,163
4,enterprise,19.31,20,680,ES,1,1743,309


In [21]:
y = data_pre["is_active"]
X = data_pre.drop(columns=["is_active"])
colums_cat = X.select_dtypes(exclude="number").columns
colums_cat

Index(['plan', 'country'], dtype='object')

In [26]:
from sklearn.impute import KNNImputer
import sklearn.impute as skl_imp
import numpy as np

# data_dummies = pd.get_dummies(X,drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_cat_encoded = encoder.fit_transform(X_train[colums_cat])
X_test_cat_encoded = encoder.transform(X_test[colums_cat])

nombres_dummies = encoder.get_feature_names_out(colums_cat)
df_train_cat = pd.DataFrame(X_train_cat_encoded, columns=nombres_dummies, index=X_train.index)
df_test_cat = pd.DataFrame(X_test_cat_encoded, columns=nombres_dummies, index=X_test.index)

X_train_num = X_train.drop(columns=colums_cat)
X_test_num = X_test.drop(columns=colums_cat)

X_train_final = pd.concat([X_train_num, df_train_cat], axis=1)
X_test_final = pd.concat([X_test_num, df_test_cat], axis=1)


imputer = SimpleImputer(strategy="mean")

X_train_imputado = imputer.fit_transform(X_train_final)
X_test_imputado = imputer.transform(X_test_final)
imputer.feature_names_in_ = X_train_final.columns.values

# MODELO

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train_imputado, y_train)

# EVALUACION

predictions = model.predict(X_test_imputado)
print(classification_report(y_test, predictions))


              precision    recall  f1-score   support

           0       0.79      0.93      0.85        40
           1       0.94      0.83      0.88        60

    accuracy                           0.87       100
   macro avg       0.87      0.88      0.87       100
weighted avg       0.88      0.87      0.87       100



In [33]:
X_test_final

,monthly_fee,support_tickets,usage_minutes,days_since_signup,days_since_last_login,plan_basic,plan_enterprise,plan_pro,country_AR,country_ES,country_FR,country_MX,country_US
361,31.41,20,1039,1668,321,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
73,22.50,10,1924,1105,118,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
374,34.26,5,347,1588,165,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
155,8.61,11,1226,561,219,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
104,41.40,15,702,789,378,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
347,32.24,13,1976,1119,373,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
86,12.75,9,1607,1254,209,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
75,44.76,0,1487,922,304,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
438,36.82,14,1716,1667,135,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [28]:
# GUARDAR MODELO
joblib.dump(model,"churn_model.pkl")
joblib.dump(imputer,"imputer.pkl")
joblib.dump(encoder,"encoder.pkl")
print("¡Los 3 archivos han sido guardados correctamente!")

¡Los 3 archivos han sido guardados correctamente!
